# Testing the RAW attention-score capture on SmolVLM2 (2.2B)

Same correctness suite as the PaliGemma notebook, but run against
**`HuggingFaceTB/SmolVLM2-2.2B-Instruct`** — a HuggingFace `transformers` VLM.

SmolVLM's source can't be edited the way the in-repo PaliGemma was, so the raw
**pre-softmax** text→vision scores are captured the *generalisable* way: we
register a custom **eager** attention function into `transformers`' attention
interface that stashes `Q Kᵀ * scaling (+mask)` before softmax (and the
post-softmax after it). The adapter fills the SAME `ProbeOutput`, so the SAME
invariant tests from `tests/test_raw_attention.py` run unchanged.

It checks the three groups of cases:
* **(A) Structure** — image/text token counts partition the sequence; the sliced
  text→vision map has exactly `n_text × n_image` elements.
* **(B) Raw-vs-softmax discriminators** — the captured tensor has negatives, its
  rows don't sum to 1, and it isn't the softmax map → it is raw logits.
* **(C) Ground-truth identity** — `softmax(raw)` reproduces the model's own
  post-softmax attention (`argmax` match + numeric reconstruction).

> **Runtime:** use a GPU runtime (`Runtime → Change runtime type → GPU`).
> SmolVLM2 is **open** (Apache-2.0) — no license/token needed.

## 1. Install dependencies
SmolVLM2 needs a recent `transformers` (the attention-interface dispatch this
capture relies on).

In [ ]:
!pip -q install -U "transformers>=4.49" accelerate huggingface_hub safetensors pillow num2words pytest

## 2. Clone the repo

In [ ]:
!rm -rf text_vision_attention_map          # force a fresh checkout (avoid a stale clone)
!git clone -q https://github.com/shubhamOjha1000/text_vision_attention_map.git
%cd text_vision_attention_map

## 3. Build the SmolVLM probe
One forward pass captures BOTH the raw pre-softmax scores and the model's own
post-softmax attention, for the first few language-model decoder layers
(bounded to keep CPU memory sane). First run downloads the weights (~4.5 GB).

In [ ]:
import importlib.util, os

# Load our test module + SmolVLM adapter by explicit path (avoids a shadowing
# top-level `tests` package on Colab's import path).
def _load(mod, rel):
    spec = importlib.util.spec_from_file_location(mod, os.path.join(os.getcwd(), rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m

T = _load("test_raw_attention", "tests/test_raw_attention.py")
S = _load("probe_smolvlm", "tests/probe_smolvlm.py")

o = S.make_smolvlm_output()          # loads SmolVLM2-2.2B and probes it
assert o is not None, "SmolVLM probe failed to load — see the printed reason above (GPU / transformers version)."
print("Probe:", o.name)
print("Sequence length L      :", o.L)
print("Decoder layers captured:", sorted(o.raw_scores))
print("raw_scores[0] shape    :", tuple(o.raw_scores[sorted(o.raw_scores)[0]].shape), "(H, L, L)")

## 4. Run ALL test cases on the real SmolVLM probe
The exact same invariant tests used for PaliGemma — unchanged.

In [ ]:
test_fns = [
    T.test_token_counts_partition_the_sequence,
    T.test_submatrix_element_count,
    T.test_raw_scores_contain_negative_values,
    T.test_raw_rows_do_not_sum_to_one,
    T.test_raw_differs_from_post_softmax,
    T.test_argmax_of_raw_matches_argmax_of_post,
    T.test_softmax_of_raw_reconstructs_post_softmax,
    T.test_within_row_log_difference_is_constant,
]

passed = 0
for fn in test_fns:
    try:
        fn(o)
        print(f"PASS  {fn.__name__}")
        passed += 1
    except AssertionError as e:
        print(f"FAIL  {fn.__name__}: {e}")

print(f"\n{passed}/{len(test_fns)} tests passed on SmolVLM2.")

## 5. (A) Structure — concrete numbers
Token counts and the sub-matrix element count. (SmolVLM's image-token count is
image-dependent, so it is not a fixed constant like PaliGemma's 256.)

In [ ]:
L0 = sorted(o.raw_scores)[0]
n_img = int(o.image_token_mask.sum())
n_txt = int(o.text_token_mask.sum())
print("image tokens :", n_img)
print("text  tokens :", n_txt)
print("image & text disjoint:", not bool((o.image_token_mask & o.text_token_mask).any()))
print("n_img + n_txt <= L    :", n_img + n_txt, "<=", o.L)

P = T.slice_text_vision(o.raw_scores[L0], o.text_token_mask, o.image_token_mask).mean(0)
print(f"\nlayer-{L0} text->vision submatrix shape:", tuple(P.shape), "(L_t, L_v)")
print("elements:", P.numel(), "==  n_txt * n_img =", n_txt * n_img)

## 6. (B) Raw-vs-softmax discriminators — concrete numbers

In [ ]:
import torch
L0 = sorted(o.raw_scores)[0]
raw0, post0 = o.raw_scores[L0], o.post_softmax[L0]
print("min raw value           :", round(raw0.min().item(), 4), " (< 0  => softmax impossible)")
print("raw row sums (first 5)  :", [round(v, 3) for v in raw0.sum(-1).flatten()[:5].tolist()],
      " (softmax rows would be 1.0)")
print("post-softmax row sum     :", round(post0.sum(-1).flatten()[0].item(), 4), " (~1.0)")
print("raw == post-softmax?     :", torch.allclose(raw0, post0, atol=1e-4), " (must be False)")

## 7. (C) Ground-truth identity — softmax(raw) == model's attention

In [ ]:
import torch
L0 = sorted(o.raw_scores)[0]
raw0 = o.raw_scores[L0].double()
post0 = o.post_softmax[L0].double()

allowed = post0 > 0
masked_raw = torch.where(allowed, raw0, torch.full_like(raw0, torch.finfo(torch.float64).min))
recon = torch.softmax(masked_raw, dim=-1)
max_diff = (recon - post0).abs()[allowed].max().item()
argmax_match = (raw0.argmax(-1) == post0.argmax(-1)).float().mean().item()

print("max |softmax(raw) - post_softmax| :", f"{max_diff:.2e}", " (~0  => exact reconstruction)")
print("argmax(raw) == argmax(post) frac  :", f"{argmax_match:.4f}", " (1.0 => same top key everywhere)")

## 8. What made this generalise?
Nothing in `tests/test_raw_attention.py` changed. We only added one adapter,
`tests/probe_smolvlm.py`, that produces a `ProbeOutput` from SmolVLM — capturing
the pre-softmax scores via the `transformers` attention interface instead of an
in-source edit. To test yet another VLM (Qwen2-VL, LLaVA-NeXT, Idefics3, ...),
write one more adapter that fills a `ProbeOutput`; every test above runs on it.